# 04 — Résumé automatique

Deux approches comparées :

1. **Résumé extractif** (TextRank via `sumy`) — rapide, sans GPU, solution par
   défaut.
2. **Résumé abstractif** (modèle HuggingFace multilingue, ex.
   `csebuetnlp/mT5_multilingual_XLSum`) — meilleure qualité rédactionnelle,
   mais plus coûteux et nécessite un téléchargement de modèle.

La fonction retenue (extractive, plus robuste et sans dépendance réseau) est
exportée vers `ged-ai/summary_tool.py` pour être réutilisée par le service
FastAPI de production.

In [1]:
import warnings
warnings.filterwarnings("ignore")

from time import time
import pandas as pd

from utils import ARTIFACTS_DIR, load_documents_csv

df = load_documents_csv("documents_extraits.csv")
print(f"{len(df)} documents chargés.")

48 documents chargés.


## 1. Résumé extractif (TextRank via sumy)

In [2]:
import nltk
for pkg in ("punkt", "punkt_tab"):
    try:
        nltk.data.find(f"tokenizers/{pkg}")
    except LookupError:
        nltk.download(pkg, quiet=True)

from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.text_rank import TextRankSummarizer


def resume_extractif(text: str, nb_phrases: int = 2) -> str:
    if not text or not text.strip():
        return ""
    parser = PlaintextParser.from_string(text, Tokenizer("french"))
    summarizer = TextRankSummarizer()
    summary = summarizer(parser.document, nb_phrases)
    return " ".join(str(s) for s in summary)

## 2. Résumé abstractif (modèle HuggingFace, optionnel)

Tentative de chargement d'un modèle de résumé abstractif multilingue. Comme
pour les embeddings (notebook 3), le téléchargement du modèle peut échouer
selon les restrictions réseau de l'environnement d'exécution — dans ce cas, le
notebook continue avec le résumé extractif seul, sans bloquer le pipeline.

**En production**, ce modèle doit être pré-téléchargé et mis en cache dans
l'image Docker du service `ged_ai_api/`, comme pour `sentence-transformers`.

In [3]:
ABSTRACTIF_DISPONIBLE = False
resume_pipeline = None

try:
    from transformers import pipeline
    resume_pipeline = pipeline(
        "summarization", model="csebuetnlp/mT5_multilingual_XLSum"
    )
    ABSTRACTIF_DISPONIBLE = True
    print("Modèle de résumé abstractif chargé.")
except Exception as e:
    print("Résumé abstractif indisponible dans cet environnement :", e)
    print("Le pipeline continue avec le résumé extractif uniquement.")


def resume_abstractif(text: str, max_len: int = 60) -> str:
    if not ABSTRACTIF_DISPONIBLE or not text.strip():
        return ""
    out = resume_pipeline(text, max_length=max_len, min_length=10, do_sample=False)
    return out[0]["summary_text"]

Résumé abstractif indisponible dans cet environnement : We couldn't connect to 'https://huggingface.co' to load the files, and couldn't find them in the cached files.
Check your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.
Le pipeline continue avec le résumé extractif uniquement.


## 3. Comparaison qualité / temps de calcul sur 10 documents

In [4]:
echantillon = df.sample(min(10, len(df)), random_state=42)

lignes = []
for _, row in echantillon.iterrows():
    text = row["texte_nettoye"]

    t0 = time()
    r_ext = resume_extractif(text, nb_phrases=2)
    t_ext = time() - t0

    t0 = time()
    r_abs = resume_abstractif(text) if ABSTRACTIF_DISPONIBLE else "(non disponible dans cet environnement)"
    t_abs = time() - t0

    lignes.append({
        "id": row["id"],
        "resume_extractif": r_ext,
        "temps_extractif_s": round(t_ext, 3),
        "resume_abstractif": r_abs,
        "temps_abstractif_s": round(t_abs, 3) if ABSTRACTIF_DISPONIBLE else None,
    })

comparaison = pd.DataFrame(lignes)
comparaison

,id,resume_extractif,temps_extractif_s,resume_abstractif,temps_abstractif_s
0,RH_synth_03.txt,Procédure d'intégration des nouveaux employés ...,0.094,(non disponible dans cet environnement),None
1,Editorial_synth_00.txt,Charte éditoriale rappelant les principes de n...,0.036,(non disponible dans cet environnement),None
2,RH_synth_02.txt,Politique de télétravail applicable aux équipe...,0.034,(non disponible dans cet environnement),None
3,Editorial_synth_03.txt,Compte-rendu de la conférence de rédaction heb...,0.036,(non disponible dans cet environnement),None
4,RH_synth_00.txt,Note d'information sur les congés payés et les...,0.037,(non disponible dans cet environnement),None
5,Technique_synth_05.txt,Guide de configuration réseau pour les postes ...,0.036,(non disponible dans cet environnement),None
6,Procedures_internes_synth_04.txt,Cette procédure décrit les étapes à suivre en ...,0.038,(non disponible dans cet environnement),None
7,Contrats_synth_03.txt,Contrat de prestation de services informatique...,0.036,(non disponible dans cet environnement),None
8,RH_sample.pdf,Note de service concernant la mise à jour du r...,0.038,(non disponible dans cet environnement),None
9,RH_synth_01.txt,Note d'information sur les congés payés et les...,0.035,(non disponible dans cet environnement),None


## 4. Export de la fonction retenue

In [5]:
# La fonction extractive est retenue par défaut : rapide, sans dépendance
# réseau, robuste en environnement contraint. Le module summary_tool.py est
# ce que le service FastAPI (ged_ai_api/) importe en production.

summary_tool_code = '''"""Fonction de résumé réutilisable par le service FastAPI (ged_ai_api/).

Résumé extractif par défaut (TextRank, sumy) : rapide, sans GPU, sans appel
réseau. Une fonction de résumé abstractif est disponible en option si un
modèle HuggingFace est pré-téléchargé et disponible localement.
"""
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.text_rank import TextRankSummarizer


def summarise_extractive(text: str, sentences_count: int = 3) -> str:
    if not text or not text.strip():
        return ""
    parser = PlaintextParser.from_string(text, Tokenizer("french"))
    summarizer = TextRankSummarizer()
    summary = summarizer(parser.document, sentences_count)
    return " ".join(str(s) for s in summary)
'''

with open("summary_tool.py", "w") as f:
    f.write(summary_tool_code)

print("summary_tool.py exporté.")

summary_tool.py exporté.


In [6]:
# Test rapide de bout en bout via le module exporté
import importlib
import summary_tool
importlib.reload(summary_tool)

exemple = df.iloc[0]["texte_nettoye"]
print("Texte original :", exemple)
print()
print("Résumé (via summary_tool.py) :", summary_tool.summarise_extractive(exemple, sentences_count=1))

Texte original : Cette procédure décrit les étapes à suivre en cas d'incident technique sur le réseau de di
ffusion. Consigne de sécurité : évacuation immédiate du bâtiment en cas d'alerte incendie,
point de rassemblement au parking nord.

Résumé (via summary_tool.py) : Consigne de sécurité : évacuation immédiate du bâtiment en cas d'alerte incendie, point de rassemblement au parking nord.


## Notes

- Le résumé extractif est retenu comme solution par défaut de production :
  aucune dépendance à un modèle téléchargé, temps de calcul très faible,
  suffisant pour les rapports d'activité internes.
- Le résumé abstractif reste une option pour les cas où la qualité
  rédactionnelle prime (ex. résumé destiné à être lu tel quel par la
  direction) — à activer une fois le modèle mis en cache dans l'image Docker
  du service IA.
- Colonnes `temps_*_s` du tableau de comparaison : à utiliser pour arbitrer
  entre les deux approches selon la volumétrie réelle de documents à résumer.